# XGBoost Men

In [1]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier # SỬ DỤNG XGBClassifier CHO CONVERSION
from sklift.models import SoloModel

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'use_label_encoder': False,
        'eval_metric': 'logloss', # Metric mặc định cho classification
        'n_jobs': -1,
        'verbosity': 0
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        params['random_state'] = SEED
        
        base_model = XGBClassifier(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        # S-Learner với Classifier trong sklift mặc định dùng predict_proba bên dưới
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (XGBoost Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="XGB_S_Learner_Conv_Tuning", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params

print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")
print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")


# 4. Đánh giá bộ tham số tốt nhất trên tập TEST
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ XGBOOST CONVERSION BEST PARAMS TRÊN TẬP TEST")
print("="*50)

test_results = []

for SEED in seeds:
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['n_jobs'] = -1
    best_params_final['verbosity'] = 0
    
    final_base_model = XGBClassifier(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    uplift_pred_test = final_s_learner.predict(X_test)
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.10f}, AUQC={auqc_score:.10f}")

results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH XGBOOST CONVERSION (9 SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

/home/datnghiemxuan/hai/.venv_3_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (XGBoost Robust Conversion)...


Best trial: 24. Best value: 0.791798: 100%|██████████| 50/50 [04:12<00:00,  5.05s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.7918
Bộ tham số Robust tốt nhất:
   n_estimators: 426
   learning_rate: 0.011994339373597094
   max_depth: 4
   min_child_weight: 16
   subsample: 0.7842622981702705
   colsample_bytree: 0.5808051306615136
   gamma: 1.0918279028788889e-06
   reg_alpha: 0.005818994716077156
   reg_lambda: 0.004985322711691849

🚀 ĐÁNH GIÁ XGBOOST CONVERSION BEST PARAMS TRÊN TẬP TEST
Seed 412312: AUUC=0.5143301252, AUQC=0.5109134159
Seed 42: AUUC=0.5029956957, AUQC=0.4999087305
Seed 1874: AUUC=0.5027858923, AUQC=0.4996839620
Seed 902745: AUUC=0.5059276046, AUQC=0.5034675843
Seed 1: AUUC=0.5009106524, AUQC=0.4979970804

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH XGBOOST CONVERSION (9 SEEDS) 🏆
**************************************************
Mean AUUC: 0.505 ± 0.005
Mean AUQC: 0.502 ± 0.005
Mean Lift: 0.005 ± 0.001
Mean KRCC: 0.020 ± 0.033


# LGBM Men

In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
from sklift.models import SoloModel

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion'
treatment_feature = 'treatment'

X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# Danh sách 5 seeds
seeds = [412312, 42, 1874, 902745, 1]

# Cố định seed cho môi trường chung (Dùng seed đầu tiên làm seed chính cho sampler)
seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbose': -1
    }
    
    val_auqc_scores = []
    
    # Duyệt qua 5 seeds BÊN TRONG hàm objective
    for SEED in seeds:
        params['random_state'] = SEED
        
        base_model = LGBMClassifier(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình của 5 seeds
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="S_Learner_Robust_Tuning", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")
print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả 5 seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1
    
    final_base_model = LGBMClassifier(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    uplift_pred_test = final_s_learner.predict(X_test)
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}")

# Tính toán điểm số trung bình trên tập Test
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds)...


Best trial: 14. Best value: 0.729516: 100%|██████████| 50/50 [08:01<00:00,  9.62s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.7295
Bộ tham số Robust tốt nhất:
   n_estimators: 145
   learning_rate: 0.001404291038988398
   max_depth: 5
   num_leaves: 12
   min_child_samples: 11
   subsample: 0.9040497977117568
   colsample_bytree: 0.5790428318847878

🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST
Seed 412312: AUUC=0.538, AUQC=0.537, Lift=0.008
Seed 42: AUUC=0.597, AUQC=0.594, Lift=0.009
Seed 1874: AUUC=0.547, AUQC=0.546, Lift=0.009
Seed 902745: AUUC=0.575, AUQC=0.573, Lift=0.010
Seed 1: AUUC=0.562, AUQC=0.560, Lift=0.008

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.564 ± 0.023
Mean AUQC: 0.562 ± 0.023
Mean Lift: 0.009 ± 0.001
Mean KRCC: 0.069 ± 0.032


# LGBM Women

In [3]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from lightgbm import LGBMClassifier
from sklift.models import SoloModel

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion'
treatment_feature = 'treatment'

X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# Danh sách 5 seeds
seeds = [412312, 42, 1874, 902745, 1]

# Cố định seed cho môi trường chung (Dùng seed đầu tiên làm seed chính cho sampler)
seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH 5 SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbose': -1
    }
    
    val_auqc_scores = []
    
    # Duyệt qua 5 seeds BÊN TRONG hàm objective
    for SEED in seeds:
        params['random_state'] = SEED
        
        base_model = LGBMClassifier(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    # Trả về AUQC trung bình của 5 seeds
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="S_Learner_Robust_Tuning", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")
print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả 5 seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST")
print("="*50)

test_results = []

for SEED in seeds:
    # Set seed cho best params
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['verbose'] = -1
    
    final_base_model = LGBMClassifier(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    uplift_pred_test = final_s_learner.predict(X_test)
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}")

# Tính toán điểm số trung bình trên tập Test
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (Tối ưu điểm trung bình 5 seeds)...


Best trial: 31. Best value: 0.812148: 100%|██████████| 50/50 [08:38<00:00, 10.36s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.8121
Bộ tham số Robust tốt nhất:
   n_estimators: 311
   learning_rate: 0.01308982709497735
   max_depth: 3
   num_leaves: 100
   min_child_samples: 181
   subsample: 0.5816016048794639
   colsample_bytree: 0.5242527018695138

🚀 ĐÁNH GIÁ BEST PARAMS TRÊN TẬP TEST
Seed 412312: AUUC=0.741, AUQC=0.741, Lift=0.002
Seed 42: AUUC=0.802, AUQC=0.802, Lift=0.006
Seed 1874: AUUC=0.775, AUQC=0.775, Lift=0.006
Seed 902745: AUUC=0.777, AUQC=0.777, Lift=0.003
Seed 1: AUUC=0.804, AUQC=0.803, Lift=0.003

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH TRÊN TẬP TEST (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.780 ± 0.025
Mean AUQC: 0.780 ± 0.025
Mean Lift: 0.004 ± 0.002
Mean KRCC: 0.145 ± 0.026


# XGB Women

In [4]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier # SỬ DỤNG XGBClassifier CHO CONVERSION
from sklift.models import SoloModel

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion' # BÀI TOÁN CONVERSION
treatment_feature = 'treatment'

X_train = train_df[in_features]
y_train = train_df[label_feature]
t_train = train_df[treatment_feature]

X_val = val_df[in_features]
y_val_true = val_df[label_feature].values.flatten()
t_val_true = val_df[treatment_feature].values.flatten()

X_test = test_df[in_features]
y_test_true = test_df[label_feature].values.flatten()
t_test_true = test_df[treatment_feature].values.flatten()

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1, 21, 2303, 23, 2007]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective (TÍNH TRUNG BÌNH SEEDS)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'use_label_encoder': False,
        'eval_metric': 'logloss', # Metric mặc định cho classification
        'n_jobs': -1,
        'verbosity': 0
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        params['random_state'] = SEED
        
        base_model = XGBClassifier(**params)
        s_learner = SoloModel(estimator=base_model)
        
        s_learner.fit(X=X_train, y=y_train, treatment=t_train)
        # S-Learner với Classifier trong sklift mặc định dùng predict_proba bên dưới
        uplift_pred_val = s_learner.predict(X_val)
        
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (XGBoost Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="XGB_S_Learner_Conv_Tuning", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ XGBOOST CONVERSION BEST PARAMS TRÊN TẬP TEST")
print("="*50)

test_results = []

for SEED in seeds:
    best_params_final = best_params.copy()
    best_params_final['random_state'] = SEED
    best_params_final['n_jobs'] = -1
    best_params_final['verbosity'] = 0
    
    final_base_model = XGBClassifier(**best_params_final)
    final_s_learner = SoloModel(estimator=final_base_model)
    final_s_learner.fit(X=X_train, y=y_train, treatment=t_train)
    
    uplift_pred_test = final_s_learner.predict(X_test)
    
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.10f}, AUQC={auqc_score:.10f}")

results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH XGBOOST CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "xgb_s_learner_conversion_robust_results_women.csv"
results_df.to_csv(csv_filename, index=False)

Loading data...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (XGBoost Robust Conversion)...


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 43. Best value: 1.00386: 100%|██████████| 50/50 [03:10<00:00,  3.81s/it] 


✅ Tuning hoàn tất! Best Mean Val AUQC: 1.0039

🚀 ĐÁNH GIÁ XGBOOST CONVERSION BEST PARAMS TRÊN TẬP TEST
Seed 412312: AUUC=0.6311743390, AUQC=0.6327646696
Seed 42: AUUC=0.5836752051, AUQC=0.5851081492
Seed 1874: AUUC=0.6634463358, AUQC=0.6633487178
Seed 902745: AUUC=0.7046457832, AUQC=0.7057466079
Seed 1: AUUC=0.6524479536, AUQC=0.6528390759
Seed 21: AUUC=0.6353053597, AUQC=0.6360967572
Seed 2303: AUUC=0.7083808989, AUQC=0.7094844272
Seed 23: AUUC=0.7002230573, AUQC=0.7006687792
Seed 2007: AUUC=0.6641398669, AUQC=0.6640623229

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH XGBOOST CONVERSION (9 SEEDS) 🏆
**************************************************
Mean AUUC: 0.660 ± 0.041
Mean AUQC: 0.661 ± 0.041
Mean Lift: 0.003 ± 0.001
Mean KRCC: 0.087 ± 0.037
